# NYC Taxi Pipeline — Analyse

Réponses aux questions analytiques du projet.  
Données : `raw.fact_taxi_trips` (janvier 2025, ~3.25M trajets) · `raw.dim_weather` · modèles dbt dans `mart.*`

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@localhost:5432/{os.getenv('POSTGRES_DB')}"
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 4)

print('Connexion OK')

---
## Partie 1 — Questions Spark

### Q1 — Distribution des durées de trajets

In [ ]:
df_duration = pd.read_sql("""
    SELECT trip_duration_minutes
    FROM raw.fact_taxi_trips
    WHERE trip_duration_minutes BETWEEN 1 AND 60
""", engine)

fig, ax = plt.subplots()
ax.hist(df_duration['trip_duration_minutes'], bins=60, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Durée (minutes)')
ax.set_ylabel('Nombre de trajets')
ax.set_title('Distribution des durées de trajets (1–60 min)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print(df_duration['trip_duration_minutes'].describe().round(2))

### Q2 — Les longs trajets reçoivent-ils plus de pourboires ?

In [ ]:
df_tip = pd.read_sql("""
    SELECT
        distance_bucket,
        ROUND(AVG(tip_percentage)::numeric, 2) AS avg_tip_pct,
        ROUND(AVG(tip_amount)::numeric, 2)     AS avg_tip_amount,
        COUNT(*)                               AS nb_trips
    FROM raw.fact_taxi_trips
    WHERE tip_percentage BETWEEN 0 AND 100
    GROUP BY distance_bucket
    ORDER BY distance_bucket
""", engine)

print(df_tip.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df_tip, x='distance_bucket', y='avg_tip_pct', ax=axes[0])
axes[0].set_title('Tip moyen (%) par tranche de distance')
axes[0].set_xlabel('Tranche de distance')
axes[0].set_ylabel('Tip moyen (%)')

sns.barplot(data=df_tip, x='distance_bucket', y='avg_tip_amount', ax=axes[1])
axes[1].set_title('Tip moyen ($) par tranche de distance')
axes[1].set_xlabel('Tranche de distance')
axes[1].set_ylabel('Tip moyen ($)')

plt.tight_layout()
plt.show()

### Q3 — Heures de prise en charge les plus chargées

In [ ]:
df_hours = pd.read_sql("""
    SELECT pickup_hour, COUNT(*) AS nb_trips
    FROM raw.fact_taxi_trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""", engine)

fig, ax = plt.subplots()
sns.barplot(data=df_hours, x='pickup_hour', y='nb_trips', ax=ax)
ax.set_title('Nombre de trajets par heure de prise en charge')
ax.set_xlabel('Heure')
ax.set_ylabel('Nombre de trajets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

peak = df_hours.loc[df_hours['nb_trips'].idxmax()]
print(f"Heure de pointe : {int(peak['pickup_hour'])}h ({int(peak['nb_trips']):,} trajets)")

### Q4 — Corrélation entre distance et pourcentage de pourboire

In [ ]:
df_corr = pd.read_sql("""
    SELECT trip_distance, tip_percentage
    FROM raw.fact_taxi_trips
    WHERE trip_distance BETWEEN 0.5 AND 25
      AND tip_percentage BETWEEN 0 AND 100
    LIMIT 50000
""", engine)

corr = df_corr['trip_distance'].corr(df_corr['tip_percentage'])
print(f"Corrélation Pearson distance / tip_percentage : {corr:.4f}")

fig, ax = plt.subplots()
ax.hexbin(df_corr['trip_distance'], df_corr['tip_percentage'],
          gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_xlabel('Distance (miles)')
ax.set_ylabel('Tip (%)')
ax.set_title(f'Distance vs Tip percentage (r = {corr:.3f})')
plt.colorbar(ax.collections[0], ax=ax, label='Nombre de trajets')
plt.tight_layout()
plt.show()

---
## Partie 2 — Questions Spark Streaming / Météo

### Q5 — Température moyenne lors des pics de trajets

In [ ]:
df_peak_temp = pd.read_sql("""
    SELECT
        t.pickup_hour,
        COUNT(*)               AS nb_trips,
        ROUND(AVG(w.temperature)::numeric, 1) AS avg_temp_c
    FROM raw.fact_taxi_trips t
    JOIN raw.dim_weather w
        ON t.pickup_hour = w.pickup_hour
       AND t.pickup_day_of_week = w.day_of_week
    GROUP BY t.pickup_hour
    ORDER BY nb_trips DESC
    LIMIT 10
""", engine)

print(df_peak_temp.to_string(index=False))

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

df_sorted = df_peak_temp.sort_values('pickup_hour')
ax1.bar(df_sorted['pickup_hour'], df_sorted['nb_trips'], alpha=0.6, label='Trajets')
ax2.plot(df_sorted['pickup_hour'], df_sorted['avg_temp_c'], color='red',
         marker='o', label='Temp. moy. (°C)')

ax1.set_xlabel('Heure')
ax1.set_ylabel('Nombre de trajets')
ax2.set_ylabel('Température moyenne (°C)')
ax1.set_title('Trajets et température par heure (top 10 heures)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Q6 — Impact du vent et de la pluie sur le nombre de trajets

In [ ]:
df_weather_impact = pd.read_sql("""
    SELECT
        w.weather_category,
        COUNT(*)                              AS nb_trips,
        ROUND(AVG(w.wind_speed)::numeric, 2)  AS avg_wind_speed,
        ROUND(AVG(t.trip_duration_minutes)::numeric, 2) AS avg_duration,
        ROUND(AVG(t.tip_percentage)::numeric, 2)        AS avg_tip_pct
    FROM raw.fact_taxi_trips t
    JOIN raw.dim_weather w
        ON t.pickup_hour = w.pickup_hour
       AND t.pickup_day_of_week = w.day_of_week
    GROUP BY w.weather_category
    ORDER BY nb_trips DESC
""", engine)

print(df_weather_impact.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label in zip(axes,
    ['nb_trips', 'avg_duration', 'avg_tip_pct'],
    ['Nombre de trajets', 'Durée moyenne (min)', 'Tip moyen (%)']):
    sns.barplot(data=df_weather_impact, x='weather_category', y=col, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('')

plt.suptitle('Impact de la météo sur les trajets', y=1.02)
plt.tight_layout()
plt.show()

---
## Partie 3 — Questions dbt / Analyse

### Q7 — Comportements de trajets selon les types de météo

In [ ]:
df_summary = pd.read_sql("""
    SELECT
        weather_category,
        SUM(total_trips)                                   AS total_trips,
        ROUND(AVG(avg_trip_duration_minutes)::numeric, 2)  AS avg_duration,
        ROUND(AVG(avg_tip_percentage)::numeric, 2)         AS avg_tip_pct,
        ROUND(AVG(avg_total_amount)::numeric, 2)           AS avg_fare
    FROM mart.trip_summary_per_hour
    WHERE weather_category IS NOT NULL
    GROUP BY weather_category
    ORDER BY total_trips DESC
""", engine)

print(df_summary.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label in zip(axes,
    ['avg_duration', 'avg_tip_pct', 'avg_fare'],
    ['Durée moyenne (min)', 'Tip moyen (%)', 'Montant moyen ($)']):
    sns.barplot(data=df_summary, x='weather_category', y=col, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('')

plt.suptitle('Comportement des trajets par catégorie météo', y=1.02)
plt.tight_layout()
plt.show()

### Q8 — À quelle heure observe-t-on le plus de zones à haute valeur ?

In [ ]:
df_hvc = pd.read_sql("""
    SELECT
        e.pickup_hour,
        COUNT(DISTINCT h.pu_location_id) AS nb_zones_hv,
        ROUND(AVG(h.avg_tip_percentage)::numeric, 2) AS avg_tip_pct
    FROM mart.high_value_customers h
    JOIN mart.trip_enriched e ON h.pu_location_id = e.pu_location_id
    GROUP BY e.pickup_hour
    ORDER BY pickup_hour
""", engine)

print(df_hvc.to_string(index=False))

fig, ax = plt.subplots()
sns.barplot(data=df_hvc, x='pickup_hour', y='nb_zones_hv', ax=ax)
ax.set_title('Zones haute valeur actives par heure')
ax.set_xlabel('Heure')
ax.set_ylabel('Nombre de zones')
plt.tight_layout()
plt.show()

### Q9 — La météo influence-t-elle les pourboires ?

In [ ]:
df_tip_weather = pd.read_sql("""
    SELECT
        weather_category,
        ROUND(AVG(tip_percentage)::numeric, 2) AS avg_tip_pct,
        ROUND(AVG(tip_amount)::numeric, 2)     AS avg_tip_amount,
        ROUND(AVG(total_amount)::numeric, 2)   AS avg_total,
        COUNT(*)                               AS nb_trips
    FROM mart.trip_enriched
    WHERE weather_category IS NOT NULL
      AND tip_percentage BETWEEN 0 AND 100
    GROUP BY weather_category
    ORDER BY avg_tip_pct DESC
""", engine)

print(df_tip_weather.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df_tip_weather, x='weather_category', y='avg_tip_pct', ax=axes[0])
axes[0].set_title('Tip moyen (%) par météo')
axes[0].set_xlabel('')
axes[0].set_ylabel('Tip moyen (%)')

sns.barplot(data=df_tip_weather, x='weather_category', y='avg_tip_amount', ax=axes[1])
axes[1].set_title('Tip moyen ($) par météo')
axes[1].set_xlabel('')
axes[1].set_ylabel('Tip moyen ($)')

plt.suptitle('Influence de la météo sur les pourboires', y=1.02)
plt.tight_layout()
plt.show()